# AWS Marketplace SaaS Product Setup
## Fixed Costs + Consumption Pricing Model

This notebook demonstrates how to create a SaaS product on AWS Marketplace with both fixed costs and consumption-based pricing.

## Step 1: Install Required Dependencies

In [ ]:
!pip install boto3 requests pandas

## Step 2: Import Libraries and Setup

In [1]:
import boto3
import json
import requests
import pandas as pd
from datetime import datetime, timedelta
import os

## Step 3: AWS Configuration

In [2]:
# AWS Configuration - Modify these as needed
AWS_REGION = 'us-east-1'  # Change to your preferred region
AWS_PROFILE = 'mp'        # Set to profile name or None for default

# Create session with optional profile
if AWS_PROFILE:
    session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
else:
    session = boto3.Session(region_name=AWS_REGION)

print(f"Using AWS Region: {AWS_REGION}")
print(f"Using AWS Profile: {AWS_PROFILE or 'default'}")

Using AWS Region: us-east-1
Using AWS Profile: mp


## Step 4: AWS Marketplace Configuration

In [3]:
# AWS Marketplace clients using session
marketplace_client = session.client('meteringmarketplace')
catalog_client = session.client('marketplace-catalog')

# Product configuration
PRODUCT_CODE = 'your-product-code'  # Replace with actual product code
DIMENSION_USAGE = 'api-calls'       # Consumption dimension
DIMENSION_FIXED = 'monthly-fee'     # Fixed cost dimension

## Step 5: Pricing Model Definition

In [4]:
# Define pricing tiers
pricing_model = {
    'fixed_costs': {
        'basic': {'monthly_fee': 0.01, 'included_calls': 10000},
        'pro': {'monthly_fee': 0.01, 'included_calls': 50000},
        'enterprise': {'monthly_fee': 999.00, 'included_calls': 200000}
    },
    'consumption': {
        'overage_rate': 0.01,  # $0.01 per API call over limit
        'bulk_discount': {
            10000: 0.008,  # $0.008 per call for 10k+ calls
            50000: 0.005   # $0.005 per call for 50k+ calls
        }
    }
}

print(json.dumps(pricing_model, indent=2))

{
  "fixed_costs": {
    "basic": {
      "monthly_fee": 0.01,
      "included_calls": 10000
    },
    "pro": {
      "monthly_fee": 0.01,
      "included_calls": 50000
    },
    "enterprise": {
      "monthly_fee": 999.0,
      "included_calls": 200000
    }
  },
  "consumption": {
    "overage_rate": 0.01,
    "bulk_discount": {
      "10000": 0.008,
      "50000": 0.005
    }
  }
}


## Step 6: Create AWS Marketplace Product

In [ ]:
def create_saas_product():
    """Create SaaS product in AWS Marketplace"""
    
    product_details = {
        "ProductTitle": "TEST psacha - My SaaS API Service contract with consumption",
        "ShortDescription": "API service with fixed and consumption pricing",
        "LongDescription": "Complete SaaS solution with tiered pricing and usage-based billing",
        "Sku": "SAAS-API-001",
        "LogoUrl": "https://awsmp-logos.s3.amazonaws.com/ca60b754fe05a24257176cdbf31c4e0d",
        "Highlights": [
            "TEST Real-time API access",
            "TEST Scalable infrastructure",
            "TEST no support"
        ],
        "Categories": ["Developer Tools", "Application Integration"],
        "SupportDescription": "Email and chat support included",
        "RefundPolicy": "30-day money back guarantee"
    }
    
    # Define pricing dimensions
    dimensions = [
        {
            "Key": "monthly-fee",
            "Description": "Monthly subscription fee",
            "Name": "Monthly Fee",
            "Unit": "Units"
        },
        {
            "Key": "api-calls",
            "Description": "API calls made to the service",
            "Name": "API Calls",
            "Unit": "Requests"
        }
    ]
    
    try:
        response = catalog_client.start_change_set(
            Catalog='AWSMarketplace',
            ChangeSet=[
                {
                    'ChangeType': 'CreateProduct',
                    'Entity': {
                        'Type': 'SaaSProduct@1.0'
                    },
                    'Details': json.dumps({
                        'ProductTitle': product_details['ProductTitle'],
                        'ShortDescription': product_details['ShortDescription'],
                        'LongDescription': product_details['LongDescription'],
                        'Sku': product_details['Sku'],
                        'LogoUrl': product_details['LogoUrl'],
                        'Highlights': product_details['Highlights'],
                        'Categories': product_details['Categories'],
                        'SupportDescription': product_details['SupportDescription'],
                        'RefundPolicy': product_details['RefundPolicy'],
                        'Dimensions': dimensions
                    })
                }
            ]
        )
        
        change_set_id = response['ChangeSetId']
        print(f"Product creation initiated. Change Set ID: {change_set_id}")
        return change_set_id
        
    except Exception as e:
        print(f"Error creating product: {e}")
        return None

In [7]:
def check_change_set_status(change_set_id):
    """Check the status of product creation"""
    try:
        response = catalog_client.describe_change_set(
            Catalog='AWSMarketplace',
            ChangeSetId=change_set_id
        )
        
        status = response['Status']
        print(f"Change Set Status: {status}")
        
        if status == 'SUCCEEDED':
            # Extract product code from the response
            for change in response['ChangeSet']:
                if 'Entity' in change and 'Identifier' in change['Entity']:
                    product_id = change['Entity']['Identifier']
                    print(f"Product created successfully! Product ID: {product_id}")
                    return product_id
        
        return None
        
    except Exception as e:
        print(f"Error checking change set: {e}")
        return None

In [11]:
# Create the product (uncomment to run)
change_set_id = create_saas_product()
if change_set_id:
    print("Waiting for product creation to complete...")
    # Check status after a few minutes
    product_code = check_change_set_status(change_set_id)
    if product_code:
        PRODUCT_CODE = product_code  # Update the global variable

print("Product creation code ready. Uncomment above lines to create actual product.")

Error creating product: 'VideoUrls'
Product creation code ready. Uncomment above lines to create actual product.


## Step 6: Usage Metering Functions

In [ ]:
def meter_usage(customer_id, dimension, quantity, timestamp=None):
    """Submit usage record to AWS Marketplace"""
    if timestamp is None:
        timestamp = datetime.utcnow()
    
    try:
        response = marketplace_client.meter_usage(
            ProductCode=PRODUCT_CODE,
            Timestamp=timestamp,
            UsageDimension=dimension,
            UsageQuantity=quantity,
            DryRun=False,
            UsageAllocations=[
                {
                    'AllocatedUsageQuantity': quantity,
                    'Tags': [
                        {
                            'Key': 'customer_id',
                            'Value': customer_id
                        }
                    ]
                }
            ]
        )
        return response
    except Exception as e:
        print(f"Error metering usage: {e}")
        return None

In [ ]:
def calculate_monthly_bill(customer_tier, api_calls_used):
    """Calculate monthly bill based on tier and usage"""
    tier_info = pricing_model['fixed_costs'][customer_tier]
    base_fee = tier_info['monthly_fee']
    included_calls = tier_info['included_calls']
    
    # Calculate overage
    overage_calls = max(0, api_calls_used - included_calls)
    overage_cost = overage_calls * pricing_model['consumption']['overage_rate']
    
    total_cost = base_fee + overage_cost
    
    return {
        'base_fee': base_fee,
        'included_calls': included_calls,
        'calls_used': api_calls_used,
        'overage_calls': overage_calls,
        'overage_cost': overage_cost,
        'total_cost': total_cost
    }

## Step 7: Example Usage and Testing

In [ ]:
# Example: Customer usage simulation
customers = [
    {'id': 'cust-001', 'tier': 'basic', 'api_calls': 15000},
    {'id': 'cust-002', 'tier': 'pro', 'api_calls': 45000},
    {'id': 'cust-003', 'tier': 'enterprise', 'api_calls': 250000}
]

# Calculate bills for each customer
billing_results = []
for customer in customers:
    bill = calculate_monthly_bill(customer['tier'], customer['api_calls'])
    bill['customer_id'] = customer['id']
    bill['tier'] = customer['tier']
    billing_results.append(bill)

# Display results
df = pd.DataFrame(billing_results)
print("Monthly Billing Summary:")
print(df.to_string(index=False))

In [ ]:
# Example: Submit usage to AWS Marketplace (dry run)
def simulate_usage_submission():
    """Simulate submitting usage data to AWS Marketplace"""
    for customer in customers:
        # Submit fixed monthly fee
        print(f"Submitting fixed fee for {customer['id']}...")
        # meter_usage(customer['id'], DIMENSION_FIXED, 1)
        
        # Submit consumption usage
        print(f"Submitting API usage for {customer['id']}: {customer['api_calls']} calls")
        # meter_usage(customer['id'], DIMENSION_USAGE, customer['api_calls'])
        
    print("Usage submission complete (simulated)")

simulate_usage_submission()

## Step 8: Marketplace Product Configuration

To complete the setup, you'll need to:

1. **Create product in AWS Marketplace Console**
2. **Define pricing dimensions:**
   - `monthly-fee`: Fixed monthly subscription
   - `api-calls`: Consumption-based API usage
3. **Set up pricing tiers** in the console
4. **Configure your SaaS application** to call the metering functions
5. **Test with AWS Marketplace Test Environment**

### Required AWS Permissions:
```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:MeterUsage",
                "aws-marketplace:BatchMeterUsage"
            ],
            "Resource": "*"
        }
    ]
}
```